# 调用agent

creat_model( ) 里面的
1.name字段，主要是为了在多agent的时候，启区分的作用
2.system_prompt字段，
3.ToolStrategy字段: 这是一个伪工具，只是为了结构化输出

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.tools import ToolRuntime, tool
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)

model=init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),

)

@tool
def get_weather(city: str) -> str:
    """天气调用
    
    Args:
        city (str): 城市名称

    Returns:
        str: 天气信息


    """
    return f"{city}天气晴朗，温度23℃"

class Person(BaseModel):
    """"用户信息"""
    name:str=Field(description="姓名")
    question:str=Field(description="问题")

class CustomState(AgentState):
    """自定义状态"""
    user:str
    
@tool
def get_user_info(runtime:ToolRuntime):
    """获取用户信息"""
    user_id=runtime.state["user"]
    return "user is John Smith " if user_id=="user_123" else "Unkonw user" 

model=create_agent(
    model=model,
    tools=[get_weather,get_user_info],
    name="agent_first",
    state_schema=CustomState,
    system_prompt="你是一名数学老师，负责帮助学生解决生活上和心里上的问题",
    response_format=ToolStrategy(
    schema=Person,
     tool_message_content="格式化输出成功",
     handle_errors=True #这是默认方式
     )

    )
response=model.invoke(
    {
        "messages": 
            [{"role": "user","content": "你好我是小代，我有一个数学问题想我您"}]
    }
    
)

stream_response=model.stream(
    {
        "messages":[
            {"role":"user","content":"我适合学习数学吗"}
        ]
    },

)


#这是invoke输出
for message in response["messages"]:
    rprint(message)

# for chunk in stream_response:
#     rprint(chunk)


工具的绑定 ：
在这里LLM和Agent有一个很重要的区别就是：LLM不会主动的调用工具，而agent会主动的调用工具  

langchain还有很多内置的工具

## 中间件 


### 1.SummarizationMiddleware
核心：在上下文快要满的时候，将一定数量的消息进行压缩总结，然后把剩下的进行保留，作为上下文

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv()
model=init_chat_model(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek:deepseek-v4-pro",
    temperature=0.5,

)

messages=[
    ("system","你是一个查询天气的助手"),
    ("human","上海今天天气如何？"),
    ("assistant","上海今天天气晴朗"),
    ("human","北京今天天气如何？"),
    ("assistant","北京今天天气阴雨"),

]

agent=create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[   #触发条件
                ("tokens",100),
                ("messages",5),
                # ("fraction",0.001)
            ],
            keep=("messages",2)  #保留的内容
        )
    ]
)
"""触发后（变成 3 条消息）：
system: （摘要内容）用户先问了上海天气，助手回答了晴天...
human:  北京今天天气如何？
assistant: 北京今天天气阴雨
"""
response=agent.invoke(
    {"messages":messages}
)


for msg in response["messages"]:
    msg.pretty_print()

###  HumanInTheLoop
HumanInTheLoop = 敏感操作前加个"停"，等人说"行"再走。 实际开发中转账、删除、下单这些场景都会用到。

### PIIMiddleware

针对个人的敏感信息：银行卡，iP地址等
可以做成hash 还有*******************5939，[id_card] 这些加密格式

###  其他中间件
前两个比较重要
1. TodoListMiddleware
适合做复杂的任务，让计划始终在上下文中

2. ModelCallLimitMiddleware
模型调用限制中间件，限制模型调用次数



-------------------

3. ToolcallLimitMiddleware
限制工具调用次数

4. ModleFallbackMiddleware
模型调用失败时，启用备用模型


5. LLMToolSelectorMiddleware
当工具太多的时候，使用子模型筛选最相关的几个工具

6. ToolRetryMiddleware
工具调用失败的时候，重新尝试

7. ModelRetryMiddleware
模型调用失败的时候，重新尝试



##  中间件的执行顺序

Middleware=[
    #中间间1，
    #中间间2，  
]


这样的执行顺序就是先1 后2 中间件

## hooks（自定义middleware）

hooks的核心特点：
----------------------------
1、不是你主动在业务代码里随便调用的
而是当流程运行到某个“钩子点”时，系统自动触发它。
2、它依附于一个更大的执行流程。比如“请求开始前”“模型调用前”“任务结束后”“异常发生时”等。
3、它的作用是让你在不改主流程源码的前提下插入自己的逻辑。例如做日志、鉴权、修改输入、拦截输出、清理资源等。

_______

hooks的分类：
1. node_style:
before_agent、 before_model、after_model、after_agent  在模型/agent执行之前/之后执行起作用

2. wrap-style:
wrap_model_call、wrap_tool_call

## 记忆

LLM本身是没有记忆能力的，langchain可以通过Memory来给LLM添加记忆能力，也就是我们说的context engineering

langchain的上下文工程是基于Agent讨论的，而是上下文工程是构建在langgraph上的

### 短期记忆
1. 基于内存的短期记忆 state(会话内部状态),checkpointer（持久化机制）,thread_id（会话作用域），主要就是下面的四步

2. 基于postgreSQL的短期记忆

______________________________
#### 注意：
1. InMemorySaver（）将状态持久化到内存，进程借宿或重建Saver（）的话，历史状态就会丢失
2. 基于外部存储介质（PostgreSQL）的持久化器，其存储的状态不会随进程终止而丢失，只要不显示删除历史状态，即可通过”thread_id"获取历史状态


In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich import print as rprint

load_dotenv()

#1.创建内存级存储器
checkpointer=InMemorySaver()  

model=init_chat_model(api_key=os.getenv("DEEPSEEK_API_KEY"),model="deepseek:deepseek-chat")


agent=create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer #2.让agent拥有了记忆功能
)

#3.让对话在同一个线程内进行，才可以记忆
config={
    "configurable":{
        "thread_id":"1"
    }
}



response=agent.invoke(  
    {"messages":[HumanMessage(content="我是张三")] },
       
        config=config  #4.传入到具体的invoke（）中
    )

response2=agent.invoke(  
    {"messages":[HumanMessage(content="我是谁？")] },
        config=config
    )

rprint(response2)

### 记忆治理策略（上下文管理）

消息的裁剪：在模型调用前
消息删除：在模型调用后
消息摘要：在模型调用后

###  长期记忆

store,namespace,key,value
_________________________________



#### 基础api的使用：
1. 基于store的api有：
put() 添加数据
get() 读取数据 返回的是一个ITEM对象，而不是value
search() 检索数据 返回的是LIST[ITEM]
2. 基于PostgresStore的api有

In [ ]:
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_openai import ChatOpenAI
from langgraph.store.memory import InMemoryStore


# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""    
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, name: str, age: int, email: str, runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, {"name": name, "age": age, "email": email})
    return "Successfully saved user info."

model = ChatOpenAI(model="gpt-5.5")

store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

###
短期记忆与长期记忆

在工具中访问长期记忆 见ppt


### 自定义短期记忆

In [ ]:

"""“““

from langchain.agents import AgentState, create_agent
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):
    user_id: str
    preferences: dict

agent = create_agent(
    "gpt-5.5",
    tools=[get_user_info],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
)

# Custom state can be passed in invoke
result = agent.invoke(
    {
        "messages": {"role": "user", "content": "Hello"},
        "user_id": "user_123",
        "preferences": {"theme": "dark"}
    },
    {"configurable": {"thread_id": "1"}})   

"””"""